In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import PyPDF2

def read_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

pdf_text = read_pdf("/content/Unit_4.pdf")

print(pdf_text[:500])


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(pdf_text)

print("Chunks:", len(chunks))


In [ ]:
!pip install -U langchain langchain-community faiss-cpu sentence-transformers


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_texts(chunks, embeddings)


In [ ]:
print("PDF text length:", len(pdf_text))
print("Number of chunks:", len(chunks))
print("Vectorstore exists:", vectorstore is not None)


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

docs = retriever.invoke("What is this PDF about?")
print("Retrieved docs:", len(docs))

for i, d in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(d.page_content[:300])


In [ ]:
from transformers import pipeline

qa = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=256
)


In [ ]:
def ask_question(question):
    docs = retriever.invoke(question)

    if not docs:
        return " Content not found in the PDF."

    context = "\n\n".join([d.page_content for d in docs])


    if len(context.strip()) < 50:
        return " Content not found in the PDF."

    prompt = f"""
You are a PDF-based assistant.

RULES:
- Answer ONLY using the given context.
- If the answer is not clearly present, say:
  "Content not found in the provided PDF."
- Give a clear, slightly detailed explanation (4–6 sentences).
- Do NOT add outside knowledge.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    result = qa(prompt)[0]["generated_text"]


    if "not found" in result.lower() or len(result.strip()) < 20:
        return " Content not found in the provided PDF."

    return f""" Answer:
{result}

 Relevant PDF Content:
{context[:800]}"""


In [ ]:
print("\nUNIT IV MEMORY AND I/O ORGANIZATION\n")
while True:
    q = input(" Ask a question (type exit to quit): ")
    if q.lower() == "exit":
        print("Thank you for using the PDF chatbot!")
        break

    print("\n Searching PDF...\n")
    response = ask_question(q)
    print(response)
    print("\n" + "="*70 + "\n")
